# Session 3: simulating a population, drift versus selection
**A. Real-world problem.** In a small population an allele can vanish or take over purely
by chance, even if it is helpful. In a big population the same allele is steadily favoured.
How small is "too small", and how much heterozygosity (genetic variety) does a population
keep over time? That question matters for conservation planning.

*General education using an idealised model. It is not a tool for real conservation
decisions, and says nothing about any person.*

**Plan.** (1) the Wright-Fisher model, (2) drift and the loss of heterozygosity,
(3) fixation probability, (4) selection and mutation updates, (5) validation against
Hardy-Weinberg and a closed form, (6) the failure case of a tiny population.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: one generation of the Wright-Fisher model
N diploid individuals have 2N alleles. Random mating (selfing allowed) means each of the
2N alleles of the next generation is an independent draw: it is A with probability p. So
the number of A alleles is Binomial(2N, p) and the new frequency is that count divided by 2N.
A seeded generator makes every run repeatable. Passing an array of p values runs many
populations at once.

In [ ]:
def drift_step(p, N, rng):
    """Next-generation allele frequency of A after sampling 2N alleles (pure drift)."""
    return rng.binomial(2 * N, p) / (2 * N)


def heterozygosity(p):
    """Expected fraction of heterozygotes 2pq under Hardy-Weinberg proportions."""
    return 2 * p * (1 - p)


def simulate_paths(N, p0, generations, runs, seed=1, s=0.0, h=0.5, mu=0.0, nu=0.0):
    """Return an array (generations + 1, runs) of allele frequencies.

    Each generation: selection, then mutation, then random sampling (drift).
    """
    rng = np.random.default_rng(seed)
    p = np.full(runs, float(p0))
    paths = [p.copy()]
    for _ in range(generations):
        p = selection_step(p, s, h)
        p = mutation_step(p, mu, nu)
        p = drift_step(p, N, rng)
        paths.append(p.copy())
    return np.array(paths)

## Step 2: selection and mutation (deterministic updates)
Fitness of the genotypes AA, Aa, aa is 1 + s, 1 + h s, 1. The mean fitness is
wbar = p^2 (1 + s) + 2 p q (1 + h s) + q^2, and after selection

    p' = [ p^2 (1 + s) + p q (1 + h s) ] / wbar

* h = 1/2: each A allele adds s/2 (additive); h = 1: A is dominant; h = 0: A is recessive.
* Simple haploid-like advantage (one allele copy per individual):
      p' = p (1 + s) / (1 + s p)
  Its closed form is logit(p_t) = logit(p_0) + t ln(1 + s), with logit(p) = ln(p / (1 - p)).

Mutation as a small constant flow: A becomes a at rate mu, a becomes A at rate nu:

    p' = p (1 - mu) + (1 - p) nu        equilibrium p* = nu / (mu + nu)

In [ ]:
def selection_step(p, s, h=0.5):
    """Diploid selection with fitness AA = 1 + s, Aa = 1 + h s, aa = 1."""
    q = 1 - p
    wbar = p * p * (1 + s) + 2 * p * q * (1 + h * s) + q * q
    return (p * p * (1 + s) + p * q * (1 + h * s)) / wbar


def haploid_selection_step(p, s):
    return p * (1 + s) / (1 + s * p)


def haploid_closed_form(p0, s, t):
    """p after t generations of haploid-like selection, from the logit formula."""
    z = np.log(p0 / (1 - p0)) + t * np.log(1 + s)
    return 1 / (1 + np.exp(-z))


def mutation_step(p, mu, nu):
    return p * (1 - mu) + (1 - p) * nu

## Step 3: what to expect, and how to check it
* Expected heterozygosity decays by a factor (1 - 1/(2N)) every generation of pure drift:
  E[H_t] = H_0 (1 - 1/(2N))^t.
* For a neutral allele the chance it eventually takes over (fixes) equals its starting
  frequency p_0.
* With a huge N and no selection, genotype frequencies stay at p^2 : 2pq : q^2.

In [ ]:
def expected_heterozygosity(H0, N, t):
    return H0 * (1 - 1 / (2 * N)) ** t


def fixation_fraction(N, p0, runs, generations, seed=1, s=0.0, h=0.5):
    """Fraction of runs where A reaches frequency 1 (and the fraction where it is lost)."""
    last = simulate_paths(N, p0, generations, runs, seed, s=s, h=h)[-1]
    return float(np.mean(last == 1.0)), float(np.mean(last == 0.0))


def random_mating_genotypes(N, p, rng):
    """Draw N children from Hardy-Weinberg proportions; return the genotype fractions (AA, Aa, aa)."""
    q = 1 - p
    return rng.multinomial(N, [p * p, 2 * p * q, q * q]) / N


def min_population_for_heterozygosity(keep, generations):
    """Smallest N with (1 - 1/(2N))^generations >= keep (mini-project)."""
    N = 1
    while (1 - 1 / (2 * N)) ** generations < keep:
        N += 1
    return N

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    gens = 100

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
    for ax, N in zip(axes, (10, 100, 1000)):
        paths = simulate_paths(N, 0.5, gens, 20)
        ax.plot(paths, lw=0.8)
        ax.set_title(f"N = {N}")
        ax.set_xlabel("generation")
        ax.grid(alpha=0.3)
        print(f"N={N}: after {gens} generations the spread (std) of p is {paths[-1].std():.3f}")
    axes[0].set_ylabel("allele frequency p of A")
    fig.suptitle("Genetic drift: 20 populations each, starting at p = 0.5", fontsize=10)
    fig.tight_layout()
    fig.savefig(OUT / "drift_paths.png", dpi=120)

    N, t = 20, 20
    paths = simulate_paths(N, 0.5, t, 20000)
    Hs = heterozygosity(paths).mean(axis=1)
    print(f"mean H after {t} generations (N={N}): simulated {Hs[-1]:.4f}, formula {expected_heterozygosity(0.5, N, t):.4f}")
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(Hs, "o", ms=3, label="simulated mean (20000 runs)")
    ax2.plot([expected_heterozygosity(0.5, N, g) for g in range(t + 1)], "-", label="H0 (1 - 1/(2N))^t")
    ax2.set_xlabel("generation")
    ax2.set_ylabel("mean heterozygosity 2pq")
    ax2.set_title("Heterozygosity decay, N = 20")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "heterozygosity_decay.png", dpi=120)

    fix, lost = fixation_fraction(10, 0.3, 4000, 400)
    print(f"neutral, N=10, p0=0.3: fixed {fix:.3f}, lost {lost:.3f} (theory: fixed 0.3)")

    # selection versus drift
    s = 0.05
    fig3, axes3 = plt.subplots(1, 2, figsize=(9, 3.8), sharey=True)
    for ax, N in zip(axes3, (10, 5000)):
        paths = simulate_paths(N, 0.05, 200, 30, s=s, h=0.5)
        ax.plot(paths, lw=0.7, alpha=0.7)
        det = [0.05]
        for _ in range(200):
            det.append(selection_step(det[-1], s, 0.5))
        ax.plot(det, "k--", lw=2, label="deterministic")
        ax.set_title(f"N = {N}, s = {s}")
        ax.set_xlabel("generation")
        ax.grid(alpha=0.3)
        ax.legend()
        fx, ls = fixation_fraction(N, 0.05, 1000, 600, s=s)
        print(f"N={N}, s={s}, p0=0.05: fixed {fx:.3f}, lost {ls:.3f}")
    axes3[0].set_ylabel("allele frequency p of A")
    fig3.suptitle("A beneficial allele (additive): lost by chance in a tiny population", fontsize=10)
    fig3.tight_layout()
    fig3.savefig(OUT / "selection_vs_drift.png", dpi=120)

    print("smallest N keeping 90% heterozygosity for 50 generations:", min_population_for_heterozygosity(0.9, 50))

## Where this stops
The model has a constant population size, equal sexes, no migration or population structure.
Real populations are messier, so treat N here as an "effective" size and the answers as rough.

## Coding tasks
1. **Easy.** Run `simulate_paths` for N = 10, 100, 1000 with the same seed and print the standard deviation of p after 50 generations. Predict the order first.
2. **Stretch.** Use `fixation_fraction` with s = 0.1 and N = 50, 500 and print how the chance of taking over changes; explain it.
3. **Explain.** Explain why a beneficial recessive allele (h = 0) spreads more slowly than a beneficial dominant one (h = 1) when it is rare.

## Assessment
- **A. Conceptual.** A helpful allele is lost in a very small population even though selection favours it. Does this mean selection did not work? Explain.
- **B. Numerical.** A population has N = 50 and a neutral allele. What fraction of heterozygosity is expected after 10 generations? Check the units and sanity.
- **C. Graph and simulation interpretation.** In the N = 10 and N = 5000 panels of the selection figure, what do the grey paths and the dashed curve tell you? What is the spread of p after 20 generations at N = 10 and N = 1000?
- **D. Design and optimisation challenge.** Design: what is the smallest N (diploid individuals) that keeps at least 90 percent of the heterozygosity for 50 generations, and how could you check it? Name two reasons the real number would be larger.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    rng = np.random.default_rng(5)
    checks.append(("edge: a lost allele (p = 0) stays lost and a fixed one (p = 1) stays fixed",
                   L.drift_step(0.0, 10, rng) == 0.0 and L.drift_step(1.0, 10, rng) == 1.0))
    a = L.simulate_paths(50, 0.4, 30, 5, seed=7)
    b = L.simulate_paths(50, 0.4, 30, 5, seed=7)
    checks.append(("a seeded simulation repeats exactly", bool(np.array_equal(a, b))))
    checks.append(("frequencies stay between 0 and 1", bool(a.min() >= 0 and a.max() <= 1)))

    # selection
    checks.append(("s = 0 leaves p unchanged", abs(L.selection_step(0.3, 0.0, 0.5) - 0.3) < 1e-12))
    p1 = L.haploid_selection_step(0.10, 0.10)
    checks.append((f"worked example: p = 0.10, s = 0.10 gives p' = 0.11 / 1.01 = 0.1089 (got {p1:.4f})", abs(p1 - 0.11 / 1.01) < 1e-12 and round(p1, 4) == 0.1089))
    p = 0.05
    for _ in range(100):
        p = L.haploid_selection_step(p, 0.05)
    checks.append(("haploid update agrees with the logit closed form after 100 generations", abs(p - L.haploid_closed_form(0.05, 0.05, 100)) < 1e-9))
    gain = [L.selection_step(0.05, 0.1, h) - 0.05 for h in (0.0, 0.5, 1.0)]
    checks.append(("at low p, a dominant advantage spreads faster than additive, and additive faster than recessive", gain[0] < gain[1] < gain[2]))
    checks.append(("hand check: diploid additive update p = 0.5, s = 0.2 gives wbar = 1.1 and p' = 0.575 / 1.1 = 0.5227",
                   abs(L.selection_step(0.5, 0.2, 0.5) - 0.5 * 1.15 / 1.1) < 1e-12))

    # mutation
    p = 0.3
    for _ in range(20000):
        p = L.mutation_step(p, 1e-3, 2e-4)
    checks.append((f"mutation alone settles at nu / (mu + nu) = 0.1667 (got {p:.4f})", abs(p - 2e-4 / 1.2e-3) < 1e-6))

    # Hardy-Weinberg with huge N and no selection
    g = L.random_mating_genotypes(10 ** 6, 0.3, np.random.default_rng(1))
    checks.append((f"huge N: genotype fractions match p^2, 2pq, q^2 = 0.09, 0.42, 0.49 (got {np.round(g, 3).tolist()})",
                   bool(np.allclose(g, [0.09, 0.42, 0.49], atol=0.003))))
    big = L.simulate_paths(10 ** 6, 0.3, 50, 20, seed=2)
    checks.append(("huge N, no selection: allele frequency barely moves in 50 generations", bool(np.abs(big[-1] - 0.3).max() < 0.01)))

    # heterozygosity decay
    paths = L.simulate_paths(20, 0.5, 20, 20000, seed=1)
    h_sim = float(L.heterozygosity(paths[-1]).mean())
    h_th = L.expected_heterozygosity(0.5, 20, 20)
    checks.append((f"worked example: heterozygosity decay (1 - 1/(2N))^t, N = 20, t = 20 gives 0.301 (formula {h_th:.4f}, simulated {h_sim:.4f})",
                   abs(h_th - 0.3013) < 5e-4 and abs(h_sim - h_th) < 0.005))
    checks.append(("hand check: N = 50, 10 generations keeps 0.99^10 = 0.9044 of H",
                   abs(L.expected_heterozygosity(1.0, 50, 10) - 0.99 ** 10) < 1e-12 and round(0.99 ** 10, 4) == 0.9044))

    # fixation probability equals starting frequency
    fix, lost = L.fixation_fraction(10, 0.3, 4000, 400)
    checks.append((f"neutral allele: fixation fraction is near the start frequency 0.3 (got {fix:.3f})", abs(fix - 0.3) < 0.03 and abs(fix + lost - 1) < 1e-9))

    # sensitivity to N
    s10 = L.simulate_paths(10, 0.5, 20, 500)[-1].std()
    s1000 = L.simulate_paths(1000, 0.5, 20, 500)[-1].std()
    checks.append((f"smaller populations drift more: spread {s10:.3f} (N=10) > {s1000:.3f} (N=1000)", s10 > 3 * s1000))
    # sensitivity to s
    fx = [L.fixation_fraction(100, 0.05, 1000, 1500, s=s)[0] for s in (0.0, 0.05, 0.2)]
    checks.append(("fixation chances at N = 100 are about 5, 37 and 86 percent for s = 0, 0.05, 0.2",
                   abs(fx[0] - 0.05) < 0.03 and abs(fx[1] - 0.37) < 0.06 and abs(fx[2] - 0.86) < 0.05))
    checks.append(("spread of p after 20 generations is about 0.4 at N = 10 and 0.05 at N = 1000", abs(s10 - 0.40) < 0.05 and abs(s1000 - 0.05) < 0.02))
    checks.append((f"a larger s raises the fixation chance at N = 100 (got {fx[0]:.2f}, {fx[1]:.2f}, {fx[2]:.2f})", fx[0] < fx[1] < fx[2]))

    # failure case
    f_small, l_small = L.fixation_fraction(10, 0.05, 1000, 600, s=0.05)
    f_big, l_big = L.fixation_fraction(5000, 0.05, 200, 600, s=0.05)
    checks.append((f"failure case: beneficial allele (s = 0.05) is lost in {l_small:.0%} of runs at N = 10", l_small > 0.8))
    checks.append((f"large N (5000): the same allele is never lost and almost always fixes (fixed {f_big:.0%})", l_big == 0 and f_big > 0.95))

    # mini-project
    n = L.min_population_for_heterozygosity(0.9, 50)
    checks.append((f"mini-project: smallest N keeping 90% heterozygosity for 50 generations is 238 (got {n})", n == 238))
    checks.append(("N = 238 passes (0.9002) and N = 237 fails (0.8998)",
                   L.expected_heterozygosity(1, 238, 50) >= 0.9 > L.expected_heterozygosity(1, 237, 50)))
    pa = L.simulate_paths(238, 0.5, 50, 5000, seed=2)
    ratio = float(L.heterozygosity(pa[-1]).mean() / 0.5)
    checks.append((f"simulation at N = 238 keeps about 90% of H (got {ratio:.3f})", abs(ratio - 0.9) < 0.02))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
